# Load Model

In [1]:
import torch
from transformers import AutoModelForCausalLM

MODEL_ID = "roneneldan/TinyStories-3M"
MODEL_REVISION = "cfaf26ec85ecdfc1bd7c2638104cce55cb67f894"

model_fp32 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float32,
)

model_fp32 = model_fp32.cpu()
model_fp32.eval()

print("Model class:", type(model_fp32).__name__)
print("Parameter dtype:", next(model_fp32.parameters()).dtype)
print("Device:", next(model_fp32.parameters()).device)

/home/randy/Documents/ACompArc/model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1011 21:16:42.905000 14901 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1011 21:16:42.927000 14901 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 108/108 [00:00<00:00, 56559.47it/s]
[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-3M
Key               

Model class: GPTNeoForCausalLM
Parameter dtype: torch.float32
Device: cpu


# Test Model

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

prompt = "Once upon a time, a little girl found a tiny dragon in her garden."

inputs = tokenizer(
    prompt,
    add_special_tokens=False,
    return_tensors="pt",
).to(model_fp32.device)

with torch.no_grad():
    generated_ids = model_fp32.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        num_beams=1,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print("Input token count:", inputs["input_ids"].shape[1])
print("\nStory:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

Input token count: 16

Story:
Once upon a time, a little girl found a tiny dragon in her garden. She was so excited and wanted to show it to her mom.

The little girl ran to her mom and said, "Mommy, look! A dragon!"

Her mom smiled and said, "Yes, it's very big and strong. But be careful, it's very dangerous."

The little girl nodded and said, "I will be careful. I will be careful


# TinyStories-3M Quantization — Fixed Specification

## Model

| Setting | Value |
|---|---|
| Hugging Face model | `roneneldan/TinyStories-3M` |
| Revision | `cfaf26ec85ecdfc1bd7c2638104cce55cb67f894` |
| Architecture | GPT-Neo |
| Transformer blocks | 8 |
| Hidden size | 128 |
| Attention heads | 16 |
| Head dimension | 8 |
| MLP dimensions | 128 → 512 → 128 |
| Reference model | Pretrained FP32 model, CPU, evaluation mode |

## Input and Generation

| Setting | Value |
|---|---|
| Batch size | 1 |
| Input sequence length | 32 tokens |
| Tokenization | Matching tokenizer; `add_special_tokens=False` |
| Input selection | First 32 token IDs; prompt must contain at least 32 tokens |
| Position IDs | 0–31 |
| Block-0 input | Token embeddings + position embeddings; dropout inactive |
| Block input/output shape | `[1, 32, 128]` |
| Generation limit | 20 additional tokens; may stop earlier at EOS |
| Decoding | Greedy: `do_sample=False`, `num_beams=1` |
| Generation cache | Enabled |
| Maximum resulting sequence length | 52 tokens |


In [3]:
# Input specification
BATCH_SIZE = 1
SEQUENCE_LENGTH = 32
GENERATED_LENGTH = 20

PROMPT = (
    "Once upon a time, a curious little girl named Lily lived in a quiet "
    "village beside a green forest. Every morning, she carried fresh bread "
    "to her grandmother and watched the birds in the trees."
)

# Verify the loaded model's architecture.
config = model_fp32.config

assert config.num_layers == 8
assert config.hidden_size == 128
assert config.num_heads == 16

HEAD_DIM = config.hidden_size // config.num_heads
MLP_HIDDEN_SIZE = config.intermediate_size or 4 * config.hidden_size

assert HEAD_DIM == 8
assert MLP_HIDDEN_SIZE == 512

BLOCK_INPUT_SHAPE = (
    BATCH_SIZE,
    SEQUENCE_LENGTH,
    config.hidden_size,
)

# Choices to apply when we construct the quantizer.
QUANTIZATION_SETTINGS = {
    "scheme": "W8A8",
    "is_per_channel": False,
    "is_dynamic": False,
    "is_qat": False,
}

# Used later for full-model generation tests.
GENERATION_SETTINGS = {
    "max_new_tokens": GENERATED_LENGTH,
    "do_sample": False,
    "num_beams": 1,
    "use_cache": True,
}

print("Block input shape:", BLOCK_INPUT_SHAPE)
print("Head dimension:", HEAD_DIM)
print("MLP hidden size:", MLP_HIDDEN_SIZE)
print("Quantization settings:", QUANTIZATION_SETTINGS)

Block input shape: (1, 32, 128)
Head dimension: 8
MLP hidden size: 512
Quantization settings: {'scheme': 'W8A8', 'is_per_channel': False, 'is_dynamic': False, 'is_qat': False}


# Saving FP32 Model From TinyStories - 3M


In [4]:
from pathlib import Path
import json

FP32_MODEL_DIR = (
    Path.home()
    / "Documents/ACompArc/model/project/artifacts/models/tinystories_3m_fp32"
)

# Check that the model parameters are still FP32.
assert all(p.dtype == torch.float32 for p in model_fp32.parameters())

# Preserve any previously saved copy.
FP32_MODEL_DIR.mkdir(parents=True, exist_ok=False)

model_fp32.save_pretrained(FP32_MODEL_DIR)
tokenizer.save_pretrained(FP32_MODEL_DIR)

metadata = {
    "model_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "dtype": "float32",
    "quantized": False,
}

(FP32_MODEL_DIR / "source_metadata.json").write_text(
    json.dumps(metadata, indent=2) + "\n"
)

print("Saved FP32 model and tokenizer to:")
print(FP32_MODEL_DIR)

print("\nSaved files:")
for path in sorted(FP32_MODEL_DIR.iterdir()):
    print(path.name)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 14.90it/s]

Saved FP32 model and tokenizer to:
/home/randy/Documents/ACompArc/model/project/artifacts/models/tinystories_3m_fp32

Saved files:
config.json
generation_config.json
model.safetensors
source_metadata.json
tokenizer.json
tokenizer_config.json


In [ ]:
# LOAD IT
# from pathlib import Path
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer

# FP32_MODEL_DIR = Path(
#     "/home/randy/Documents/ACompArc/model/project/"
#     "artifacts/models/tinystories_3m_fp32"
# )

# # Load the saved FP32 model.
# model_fp32 = AutoModelForCausalLM.from_pretrained(
#     FP32_MODEL_DIR,
#     local_files_only=True,
#     dtype=torch.float32,
# )
# model_fp32 = model_fp32.cpu()
# model_fp32.eval()

# # Load its matching tokenizer.
# tokenizer = AutoTokenizer.from_pretrained(
#     FP32_MODEL_DIR,
#     local_files_only=True,
# )

# print("Model:", type(model_fp32).__name__)
# print("Weight dtype:", next(model_fp32.parameters()).dtype)
# print("Device:", next(model_fp32.parameters()).device)
# print("Transformer blocks:", len(model_fp32.transformer.h))